In [1]:
import cv2 as cv
import numpy as np
import mediapipe as mp
from collections import deque, Counter
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.resnet50 import preprocess_input
import time

# ---------------- CONFIG ----------------
MODEL_PATH = "resnet50_asl_augmented_frozen.h5"
CONF_THRESHOLD = 0.6               # ignore weak predictions
SMOOTH_WINDOW = 8                  # frames for smoothing
MARGIN = 20                        # small safe margin around hand

# Label mapping (index → class)
CLASS_LABELS = {
     0:'A',  1:'B',  2:'C',  3:'D',  4:'E',  5:'F',  6:'G',  7:'H',  8:'I',  9:'J',
    10:'K', 11:'L', 12:'M', 13:'N', 14:'O', 15:'P', 16:'Q', 17:'R', 18:'S', 19:'T',
    20:'U', 21:'V', 22:'W', 23:'X', 24:'Y', 25:'Z', 26:'del', 27:'nothing', 28:'space'
}

# ---------------- LOAD MODEL ----------------
print("🔹 Loading model...")
model = load_model(MODEL_PATH)
IMG_H, IMG_W, IMG_C = model.input_shape[1:4]
print(f"✅ Model loaded successfully ({IMG_H}x{IMG_W}x{IMG_C})")

# ---------------- INIT MEDIAPIPE ----------------
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.6)

# ---------------- SMOOTHING BUFFER ----------------
pred_history = deque(maxlen=SMOOTH_WINDOW)

# ---------------- CAMERA ----------------
cap = cv.VideoCapture(0)
ptime = 0
print("🎥 Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Frame read failed.")
        break

    frame = cv.flip(frame, 1)
    h, w, _ = frame.shape

    # 🎨 Convert to gray for display
    gray_frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    gray_frame = cv.cvtColor(gray_frame, cv.COLOR_GRAY2BGR)

    # 🧠 Process hands
    rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    results = hands.process(rgb)

    pred_label, conf = "nothing", 0.0

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get bounding box
            x = [lm.x for lm in hand_landmarks.landmark]
            y = [lm.y for lm in hand_landmarks.landmark]
            x_min, y_min = int(min(x) * w), int(min(y) * h)
            x_max, y_max = int(max(x) * w), int(max(y) * h)

            # Add margin, clip to frame
            x_min = max(0, x_min - MARGIN)
            y_min = max(0, y_min - MARGIN)
            x_max = min(w, x_max + MARGIN)
            y_max = min(h, y_max + MARGIN)

            # Crop hand region
            hand_roi = frame[y_min:y_max, x_min:x_max]
            if hand_roi.size == 0:
                continue

            # Preprocess for ResNet50
            hand_img = cv.resize(hand_roi, (IMG_W, IMG_H))
            hand_img = cv.cvtColor(hand_img, cv.COLOR_BGR2RGB)
            img_arr = preprocess_input(np.expand_dims(hand_img.astype("float32"), axis=0))

            # Predict
            preds = model.predict(img_arr, verbose=0)[0]
            conf = float(np.max(preds))
            label_idx = int(np.argmax(preds))
            pred_label = CLASS_LABELS[label_idx] if conf >= CONF_THRESHOLD else "nothing"

            # Add to smoothing buffer
            pred_history.append(pred_label)

            # Draw box and landmarks
            cv.rectangle(gray_frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            cv.putText(gray_frame, f"{pred_label} ({conf*100:.1f}%)",
                       (x_min + 5, y_min - 10),
                       cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            # Draw hand connections
            for connection in mp_hands.HAND_CONNECTIONS:
                start, end = connection
                s, e = hand_landmarks.landmark[start], hand_landmarks.landmark[end]
                cv.line(gray_frame, (int(s.x*w), int(s.y*h)), (int(e.x*w), int(e.y*h)), (255, 255, 255), 2)
            for lm in hand_landmarks.landmark:
                cx, cy = int(lm.x*w), int(lm.y*h)
                cv.circle(gray_frame, (cx, cy), 4, (0, 0, 255), -1)

    # 🧩 Smoothed stable prediction
    stable_label = Counter(pred_history).most_common(1)[0][0] if pred_history else "nothing"

    # Display final stable prediction
    cv.putText(gray_frame, f"Stable: {stable_label}",
               (10, 40), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # FPS Counter
    ctime = time.time()
    fps = 1 / (ctime - ptime) if ptime else 0
    ptime = ctime
    cv.putText(gray_frame, f"FPS: {fps:.1f}", (10, h - 20),
               cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # Display frame
    cv.imshow("ASL Detection (Perfect Gray Mode)", gray_frame)
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()
print("👋 Detection ended.")


🔹 Loading model...


✅ Model loaded successfully (224x224x3)
🎥 Press 'q' to quit.
👋 Detection ended.
